In [3]:
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
import plotly.graph_objects as go
import random
import warnings
warnings.filterwarnings('ignore')

print("all good")

all good


In [4]:
# Task 1: Data Preparation & Validation

# Load the three files
features = pd.read_csv("elliptic_txs_features.csv", header=None)
classes  = pd.read_csv("elliptic_txs_classes.csv")
edges    = pd.read_csv("elliptic_txs_edgelist.csv")

# The features file has no column names — assign them
# Column 0 = transaction ID, Column 1 = time step, Columns 2-166 = 165 anonymised features
feature_cols = ['txId', 'time_step'] + [f'feature_{i}' for i in range(1, 166)]
features.columns = feature_cols

print("Features shape:", features.shape)
print("Classes shape: ", classes.shape)
print("Edges shape:   ", edges.shape)

Features shape: (203769, 167)
Classes shape:  (203769, 2)
Edges shape:    (234355, 2)


In [5]:
# Merge features and classes on txId
df = features.merge(classes, on='txId', how='left')

print("Merged shape:", df.shape)
print("\nClass distribution:")
print(df['class'].value_counts())

print("\nMissing values per column (showing non-zero only):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing[missing > 0].any() else "No missing values found")

print("\nDuplicate txIds:", df.duplicated(subset='txId').sum())

print("\nSelf-loops in edges (txId1 == txId2):", (edges['txId1'] == edges['txId2']).sum())

Merged shape: (203769, 168)

Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Missing values per column (showing non-zero only):
No missing values found

Duplicate txIds: 0

Self-loops in edges (txId1 == txId2): 0


In [6]:
# Standardise class labels to readable values
# 1 = illicit, 2 = licit, unknown stays as unknown
df['class'] = df['class'].replace({'1': 'illicit', '2': 'licit', 1: 'illicit', 2: 'licit'})

print("Standardised class distribution:")
print(df['class'].value_counts())

# Save cleaned dataset
df.to_csv("elliptic_cleaned.csv", index=False)
print("\nCleaned dataset saved as elliptic_cleaned.csv")

Standardised class distribution:
class
unknown    157205
licit       42019
illicit      4545
Name: count, dtype: int64

Cleaned dataset saved as elliptic_cleaned.csv


In [7]:
# Check how many transactions have no edges
nodes_in_edges = set(edges['txId1']).union(set(edges['txId2']))
nodes_in_features = set(df['txId'])

isolated = nodes_in_features - nodes_in_edges
print(f"Transactions with no edges: {len(isolated)}")
print(f"Transactions with at least one edge: {len(nodes_in_edges)}")

Transactions with no edges: 0
Transactions with at least one edge: 203769


In [8]:
# Check data types
print("Feature dtypes (unique):")
print(df.dtypes.value_counts())

# Check time step values — should only be integers 1 to 49
print("\nTime step range:")
print(f"Min: {df['time_step'].min()}, Max: {df['time_step'].max()}")
print(f"Unique time steps: {sorted(df['time_step'].unique())}")

# Check for any non-numeric values in feature columns
feature_columns = [f'feature_{i}' for i in range(1, 166)]
non_numeric = df[feature_columns].apply(pd.to_numeric, errors='coerce').isnull().sum().sum()
print(f"\nNon-numeric values in feature columns: {non_numeric}")

# Check class column contains only expected values
print("\nUnique class values:")
print(df['class'].unique())

# Check edge columns contain only integers
print("\nEdge txId1 dtype:", edges['txId1'].dtype)
print("Edge txId2 dtype:", edges['txId2'].dtype)

# Check for negative values in time step
print("\nNegative time steps:", (df['time_step'] < 1).sum())
print("Time steps above 49:", (df['time_step'] > 49).sum())

Feature dtypes (unique):
float64    165
int64        2
object       1
Name: count, dtype: int64

Time step range:
Min: 1, Max: 49
Unique time steps: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49)]

Non-numeric values in feature columns: 0

Unique class values:
['unknown' 'licit' 'illicit']

Edge txId1 dtype: int64
Edge txId2 dtype: int64

Negative time steps: 0
Time s

In [9]:
ghost_nodes = nodes_in_edges - nodes_in_features
print(f"Nodes in edges but not in features: {len(ghost_nodes)}")

Nodes in edges but not in features: 0


In [10]:
# Aggregate edge weights
# If transaction A sends to transaction B multiple times,
# those appear as multiple rows — we count them as edge weight
edges_weighted = (
    edges
    .groupby(['txId1', 'txId2'])
    .size()
    .reset_index(name='weight')
)

print(f"Original edges:   {len(edges)}")
print(f"Weighted edges:   {len(edges_weighted)}")
print(f"Max weight:       {edges_weighted['weight'].max()}")
print(f"Edges with weight > 1: {(edges_weighted['weight'] > 1).sum()}")

# Replace edges with weighted version
edges = edges_weighted

Original edges:   234355
Weighted edges:   234355
Max weight:       1
Edges with weight > 1: 0


## Task 1: Data Preparation & Validation — Documentation

### Decisions and Business Justifications

**Left join on merge**
A left join was used when merging features and classes to preserve all 203,769 transactions
regardless of whether they have a class label. Dropping unlabelled transactions would remove
157,205 nodes (77% of the dataset) and destroy the graph structure needed for network analysis.

**No missing values — no action required**
The merged dataset contains zero null values across all 168 columns. No imputation or row
removal was necessary.

**No duplicates — no action required**
Zero duplicate transaction IDs were found. Each node in the graph is uniquely represented.

**No self-loops — no action required**
Zero self-loops were found in the edge list. No transaction references itself as a payment
destination, so no edges needed to be removed.

**Class label standardisation**
Labels were stored as integers (1, 2) and strings ('1', '2') inconsistently. Both forms were
mapped to human-readable labels: 1/'1' to 'illicit', 2/'2' to 'licit', 'unknown' unchanged.
This prevents silent errors during filtering and visualisation downstream.

**Ghost node validation**
Checked for transaction IDs present in the edge list but absent from the features dataset.
Zero ghost nodes found. Every node NetworkX creates from the edge list has a corresponding
feature row and class label. No silent null nodes will exist in the graph.

**Isolated node validation**
Checked for transactions in the features dataset with no edges. Zero isolated nodes found.
All 203,769 transactions participate in at least one edge, meaning the full dataset is
represented in the graph without any disconnected nodes.

**Edge weight aggregation**
The edge list was aggregated by (txId1, txId2) pairs to compute edge weights as required
by Task 2. All 234,355 edges have a weight of 1, confirming no two transactions connected
more than once. This is consistent with Bitcoin's transaction model where each payment is
a unique blockchain event with a unique transaction ID.

**Data type validation**
All 165 feature columns confirmed as numeric (float64). Time step column confirmed as
integer with values strictly between 1 and 49. No out-of-range or incorrectly typed values
found across any column.

**Memory optimisation — skipped**
Downcasting float64 to float32 was considered but rejected. The features dataframe is not
used in graph computations — NetworkX builds the graph from the edge list only. The
precision cost outweighs the negligible RAM benefit for this specific workload.

### Final Dataset Summary
- Transactions: 203,769
- Edges: 234,355 (all weights = 1)
- Illicit: 4,545 (2.2%)
- Licit: 42,019 (20.6%)
- Unknown: 157,205 (77.1%)
- Missing values: 0
- Duplicates: 0
- Self-loops: 0
- Ghost nodes: 0
- Isolated nodes: 0

In [11]:
# Task 2: Network Construction
# Build a directed graph where each node is a transaction
# and each edge represents Bitcoin flowing from one transaction to another

# Create lookup dictionaries for fast node attribute access
class_map    = df.set_index('txId')['class'].to_dict()
timestep_map = df.set_index('txId')['time_step'].to_dict()

# Build the full directed graph from the weighted edge list
G_full = nx.from_pandas_edgelist(
    edges_weighted,
    source='txId1',
    target='txId2',
    edge_attr='weight',
    create_using=nx.DiGraph()
)

# Attach class label and time step as node attributes
nx.set_node_attributes(G_full, class_map, 'label')
nx.set_node_attributes(G_full, timestep_map, 'time_step')

print(f"Nodes: {G_full.number_of_nodes():,}")
print(f"Edges: {G_full.number_of_edges():,}")
print(f"Is directed: {G_full.is_directed()}")

Nodes: 203,769
Edges: 234,355
Is directed: True


In [14]:
# Compute approximate betweenness centrality using a 20,000 node sample
print("Computing betweenness centrality (this may take a minute or two)...")
approx_betweenness = nx.betweenness_centrality(G_full, k=20000, seed=42)

# Sort nodes by betweenness score descending
sorted_nodes = sorted(approx_betweenness, key=approx_betweenness.get, reverse=True)

# Top 50 hubs by betweenness centrality
top_hubs = sorted_nodes[:50]

# Extract ego network for each hub — the hub plus all direct neighbours
ego_graphs = [nx.ego_graph(G_full, node) for node in top_hubs]

# Combine all 50 ego graphs into one working subgraph
G_sub = nx.compose_all(ego_graphs)

# Safety check — enforce 5,000 node minimum
current_hub_index = 50
while G_sub.number_of_nodes() < 5000 and current_hub_index < len(sorted_nodes):
    next_hub = sorted_nodes[current_hub_index]
    next_ego = nx.ego_graph(G_full, next_hub)
    G_sub = nx.compose(G_sub, next_ego)
    current_hub_index += 1

print(f"Final Subgraph Nodes: {G_sub.number_of_nodes():,}")
print(f"Final Subgraph Edges: {G_sub.number_of_edges():,}")
print(f"Number of hubs used: {current_hub_index}")

Computing betweenness centrality (this may take a minute or two)...
Final Subgraph Nodes: 5,000
Final Subgraph Edges: 5,635
Number of hubs used: 4462


In [ ]:
import pickle

# Save the betweenness scores and subgraph
with open("approx_betweenness.pkl", "wb") as f:
    pickle.dump(approx_betweenness, f)

nx.write_graphml(G_sub, "subgraph.graphml")

print("Betweenness scores saved to approx_betweenness.pkl")
print("Subgraph saved to subgraph.graphml")

Betweenness scores saved to approx_betweenness.pkl
Subgraph saved to subgraph.graphml


In [16]:
# TO RELOAD WITHOUT RERUNNING (if kernel crashes):
# with open("approx_betweenness.pkl", "rb") as f:
#     approx_betweenness = pickle.load(f)
# G_sub = nx.read_graphml("subgraph.graphml")
# sorted_nodes = sorted(approx_betweenness, key=approx_betweenness.get, reverse=True)
# top_hubs = sorted_nodes[:50]